In [1]:
"""
Makes a catalogue of morphological data for my galaxies. Creates fits data file Galaxy_morphology.
"""

from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
import corner
import asdf
from tqdm import tqdm
from photutils.aperture import EllipticalAperture

with fits.open("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits") as hdul:
        data = hdul[1].data
TABLE = Table(data)


GALAXY_ID = TABLE["SURVEY_ID"]        # object ID, used for labeling/output files
SERSIC_FILTERS = ["F444W","F356W","F277W"]                    # filter name, used for labeling/output files
SURVEY = TABLE["SURVEY"]

CUTOUT_SIZE = 0.96 #as
PIXEL_SIZE = 0.03 #as



In [96]:
def read_summary_table(path, parameter):
    """
    Extracts the mean and sd values for a given parameter from the sersic summary tables.
    """
    csv_table = pd.read_csv(path, index_col=0)
    
    row = csv_table.loc[parameter]
    return row["mean"], row["sd"]

In [97]:
def read_residual_fits_table(path):
    """
    Opens data_model_residual fits files and returns data for sersic model and residual.
    """
    with fits.open(path) as hdul:
        sci_im = hdul[1].data
        sersic_model = hdul[2].data
        residual = hdul[3].data
        mask = hdul[4].data
        mask = mask.astype(bool)
        rms = hdul[5].data

    return sci_im, sersic_model, residual, mask, rms

In [ ]:
def calculate_RFF(sci_im, residual, mask, rms, flux_auto, flux_radius):
    """
    Formula for RFF from EPOCHS XI eq 4.

    Parameters
    ----------
    sci_im : array
        science image
    residual : array
        residual data (between sersic model and science image)
    mask : array
        mask map
    rms : array
        map of background noise
    flux_auto : float
        Flux of galaxy measured through SExtractor
    flux_radius : float
        Half-light radius measurement measured with SExtractor
    """

    ny, nx = sci_im.shape
    y0, x0 = ny // 2, nx // 2

    y, x = np.indices(sci_im.shape)

    r = np.sqrt((x - x0)**2 + (y - y0)**2)

    rff_region = r <= 2 * flux_radius
    # Defines what region is the galaxy and therefore where RFF can be meaningfully calculated

    # Remove contaminating sources
    good_pixels = rff_region & (~mask)

    # Number of pixels in aperture (RFF is calculated within twice flux_radius)
    N_pixels = np.sum(good_pixels)

    background_values = sci_im[~mask.astype(bool)]
    depth_1sig = 1.4826 * np.nanmedian(np.abs(background_values - np.nanmedian(background_values)))

    rff = ( np.sum(np.abs(residual[good_pixels])) - 0.8 * depth_1sig * N_pixels)  / flux_auto
    # Must restrict the residual map to the region where RFF is defined (twice flux_radius)

    return rff

In [99]:
def _calc_RFF(sci_im, model, residuals, mask, a_image_as, b_image_as, theta_image):
        """
        Calculates the Residual Flux Fraction (RFF) for a given galaxy.

        Parameters
        ----------
        model : array
            Sersic model data
        residuals : array
            Residuals bewteen science image and model
        mask : array of bool
            Mask map
        a_image_as : float
            Semi major axis of elliptical aperture used in image (units of as)
        b_image_as : float
            Semi minor axis of elliptical aperture used in image (units of as)
        theta_image : float
            Angle of orientation of elliptical aperture used in image (degrees?)
        """
        # -> Kron radius too small
        # -> elliptical aperture also too small

        pix_scale = CUTOUT_SIZE / PIXEL_SIZE
        # reconstruct Kron elliptical aperture
        # EllipticalAperture models an elliptical aperture taking 5 parameters: x0, y0, a, b, orientation
        kron_aper = EllipticalAperture(
            (PIXEL_SIZE / 2, PIXEL_SIZE / 2),
            a_image_as / pix_scale,
            b_image_as / pix_scale,
            theta_image,
        )
        residuals = np.abs(residuals) 
        model_kron = kron_aper.do_photometry(model)[0][0]
        residual_kron = kron_aper.do_photometry(residuals)[0][0]

        # Mask all sources except those within Kron aperture
        mask[mask != 0] = 1
        background_values = sci_im[~mask.astype(bool)]
        abs_deviation = np.abs(background_values - np.nanmedian(background_values))
        depth_1sig = 1.4826 * np.nanmedian(abs_deviation)

        rff = (residual_kron - 0.8 * depth_1sig * kron_aper.area) / model_kron
        return rff

In [ ]:
def calculate_BIC(sci_im, residual, mask, rms, flux_radius, n_params):
    """
    Calculates BIC, assuming Gaussian pixel uncertainties.
    """
    ny, nx = sci_im.shape
    y0, x0 = ny // 2, nx // 2

    y, x = np.indices(sci_im.shape)

    r = np.sqrt((x - x0)**2 + (y - y0)**2)

    region = r <= 2 * flux_radius
    # Defines what region is the galaxy and therefore where RFF can be meaningfully calculated
    # TODO: I'm not sure if this is appropriate for the BIC

    # Remove contaminating sources and ensure that rms is not Nan nor zero
    good_pixels = region & (~mask) & np.isfinite(rms) & (rms != 0)

    # Number of pixels in aperture (RFF is calculated within twice flux_radius)
    N_pixels = np.sum(good_pixels)

    err = rms[good_pixels]
    # log-likelihood for Gaussian pixel uncertainties
    # TODO: not sure if thats appropriate here too
    ln_L = -0.5 * np.sum( residual[good_pixels]**2 / err**2 + np.log(2 * np.pi * err**2) )

    bic = n_params * np.log(N_pixels) - 2 * ln_L

    return bic
    

In [100]:
def make_single_sersic_table():
    """
    Makes a FITS table of morphological parameters for each galaxy.
    Each galaxy occupies a single row, with separate columns for each filter.
    """

    rows = []

    for i in tqdm(range(len(GALAXY_ID)), total=len(GALAXY_ID)):

        # Start the row with the galaxy information
        row = {
            "SURVEY_ID": GALAXY_ID[i],
            "SURVEY": SURVEY[i],
            "REDSHIFT": TABLE["REDSHIFT"][i],
        }

        # Add morphology measurements for each filter
        for filt in SERSIC_FILTERS:

            summary_path = Path(
                f"/nvme/scratch/work/alberttg/Summer_project/Single_sersic_fits/"
                f"{GALAXY_ID[i]}/{GALAXY_ID[i]}_{filt}_summary.csv"
            )

            if summary_path.exists():
                ellip_mean, ellip_sd = read_summary_table(summary_path, "ellip")
                n_mean, n_sd = read_summary_table(summary_path, "n")
                r_eff_mean, r_eff_sd = read_summary_table(summary_path, "r_eff")
            else:
                ellip_mean = ellip_sd = np.nan
                n_mean = n_sd = np.nan
                r_eff_mean = r_eff_sd = np.nan

            row[f"{filt}_ellip_mean"] = ellip_mean
            row[f"{filt}_ellip_sd"] = ellip_sd
            row[f"{filt}_n_mean"] = n_mean
            row[f"{filt}_n_sd"] = n_sd
            row[f"{filt}_r_eff_mean"] = r_eff_mean
            row[f"{filt}_r_eff_sd"] = r_eff_sd

            fits_path = Path(f"/nvme/scratch/work/alberttg/Summer_project/Single_sersic_fits/{GALAXY_ID[i]}/{GALAXY_ID[i]}_{filt}_data_model_residual.fits")

            if fits_path.exists():
                sci_im, sersic_model, residual, mask, rms \
                    = read_residual_fits_table(fits_path)
                
                flux_auto = f"FLUX_AUTO_{filt}"
                flux_radius = f"FLUX_RADIUS_{filt}"
                a_image = f"A_IMAGE_{filt}"
                b_image = f"B_IMAGE_{filt}"
                theta_image = f"THETA_IMAGE_{filt}"
                
                my_rff = calculate_RFF(sci_im, residual, mask, rms, TABLE[flux_auto][i], TABLE[flux_radius][i])

                rff = _calc_RFF(sci_im, sersic_model, residual, mask, TABLE[a_image][i], TABLE[b_image][i], TABLE[theta_image][i])

            else:
                rff = np.nan
                my_rff = np.nan

            row[f"{filt}_my_RFF"] = my_rff
            row[f"{filt}_RFF"] = rff

        # Append one completed row per galaxy
        rows.append(row)

    new_table = Table(rows=rows)

    print(new_table.colnames)

    new_table.write("All_galaxies_single_sersic_data.fits", format="fits", overwrite=True)

    return new_table

In [ ]:
if __name__ == "__main__":
    table = make_single_sersic_table()
    
    

  2%|▏         | 3/144 [00:00<00:05, 24.16it/s]

100%|██████████| 144/144 [00:03<00:00, 45.94it/s]


['SURVEY_ID', 'SURVEY', 'REDSHIFT', 'F444W_ellip_mean', 'F444W_ellip_sd', 'F444W_n_mean', 'F444W_n_sd', 'F444W_r_eff_mean', 'F444W_r_eff_sd', 'F444W_my_RFF', 'F444W_RFF', 'F356W_ellip_mean', 'F356W_ellip_sd', 'F356W_n_mean', 'F356W_n_sd', 'F356W_r_eff_mean', 'F356W_r_eff_sd', 'F356W_my_RFF', 'F356W_RFF', 'F277W_ellip_mean', 'F277W_ellip_sd', 'F277W_n_mean', 'F277W_n_sd', 'F277W_r_eff_mean', 'F277W_r_eff_sd', 'F277W_my_RFF', 'F277W_RFF']
nan
nan
nan
